In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pyam
from pathlib import Path
import os
# import pandas as pd
from pandas_indexing import *

from ikea_pathways.data_check import siamese_data_check
import data_shepherd as ds
import matplotlib.pyplot as plt

<IPython.core.display.Javascript object>

In [3]:
def elec(tech=None):
    if not tech is None:
        return f'Secondary Energy|Electricity|{tech}'
    else:
        return 'Secondary Energy|Electricity'

In [4]:
# REMIND_RMAP = ds.utils.RegionMapping.from_model('REMIND_2.1')
# REMIND_RMAP

In [5]:
IKEA_ISOS = ["DEU","POL","BRA","MEX","KEN","MAR","MOZ",#"EU27",
             "NGA","SEN","ZAF","USA","NAM","DZA","TUR","SAU",
             "ARE","BGD","IND","IDN","PAK","VNM","GBR","AUS","CHN","JPN"]

IKEA_ISOS.sort()

# Load data

In [6]:
BOX_MOUNT_PATH = Path("~/Library/CloudStorage/Box-Box").expanduser()
if not BOX_MOUNT_PATH.is_dir():
    BOX_MOUNT_PATH = Path("~/Box").expanduser()

In [7]:
DSCALE_PATH = (
    "Climate Policy Team/"
    "02 - Projects/"
    "IKEA NDC 1.5° Pathways 23-25 - phase II/"
    "2 - Work Packages/"
    "WP2 - National 1.5°C Pathways/"
    "Downscaling/"
    "DSCALE"
)

DSCALE_PATH: os.PathLike = BOX_MOUNT_PATH / DSCALE_PATH

In [8]:
dscale_project = 'REMIND_fuel_mix_testing'
dscale_suffix = '19_02_2026'

In [9]:
# Load the most recent data
dscale_results = pyam.IamDataFrame(f'../results/5_Explorer_and_New_Variables/{dscale_project}/{dscale_suffix}/REMIND 3.4_2022_harmo_step5h_hydrogen.csv')

pyam - INFO: Running in a notebook, setting up a basic logging at level INFO
pyam.core - INFO: Reading file ../results/5_Explorer_and_New_Variables/REMIND_fuel_mix_testing/19_02_2026/REMIND 3.4_2022_harmo_step5h_hydrogen.csv


In [10]:
# Enforcing regional_consistency on the fuels (fabio to show where this can be done)
# Enforcing that fuels can never be greater than carriers (Fabio to show this)
# Allowing an "other" variable to be reported (Fabio to show)

# Perform standard NPE data checks
This will check PE/FE and SE/FE consistency across fuels, making sure the sum of one is not bigger than another. 

It will also highlight any important spike or drop in Emissions data and in Fossil Fuel data. 

In [11]:
class config:
    data_check_path : os.PathLike = DSCALE_PATH / "data_checks"

In [12]:
siamese_data_check(dscale_results.filter(region=IKEA_ISOS), config)

Coal
Gas
Oil
Biomass
Siamese check completed.


# Rogue variable check

* Carbon Sequestration variables: computed in Step3 
* Emissions variables: We should have CO2 emissions for Energy and Energy sub-sector (except Transport and Power atm), as well as for IPPU and LULUCF. GHG total, and Non-CO2 data.
* Final Energy: by carrier (Liquids, Solids, Gases) and by subsector (Industry, Transportation and Buildings)
* Secondary Energy: for Electricity, but also for Gases, Liquids, Solids (and fuel mix), and Hydrogen and Heat
* Primary Energy: total and fuel mix
* Prices: derived from the IAMs directly - not downscaled per say
* Tax Revenues
* Trade of FF
* Statistical difference for Emissions, PE and SE
* Population and GDP

In [13]:
# Careful to filter for a downscaled country, for regional countries (USA, IND, JPN), more variables will be available, directly copied from the IAMs results.
dscale_results.filter(region = "BRA").variable

['Carbon Sequestration|CCS',
 'Carbon Sequestration|CCS|Biomass',
 'Carbon Sequestration|CCS|Fossil',
 'Carbon Sequestration|CCS|Industrial Processes',
 'Emissions|CO2',
 'Emissions|CO2|Energy',
 'Emissions|CO2|Energy|Demand|Industry',
 'Emissions|CO2|Energy|Demand|Residential and Commercial',
 'Emissions|CO2|Industrial Processes',
 'Emissions|CO2|LULUCF Direct+Indirect',
 'Emissions|CO2|LULUCF Indirect',
 'Emissions|Kyoto Gases (incl. indirect AFOLU)',
 'Emissions|Total Non-CO2',
 'Final Energy',
 'Final Energy|Electricity',
 'Final Energy|Gases',
 'Final Energy|Heat',
 'Final Energy|Industry',
 'Final Energy|Industry|Electricity',
 'Final Energy|Industry|Gases',
 'Final Energy|Industry|Gases|Biomass',
 'Final Energy|Industry|Gases|Electricity',
 'Final Energy|Industry|Gases|Natural Gas',
 'Final Energy|Industry|Heat',
 'Final Energy|Industry|Hydrogen',
 'Final Energy|Industry|Liquids',
 'Final Energy|Industry|Liquids|Biomass',
 'Final Energy|Industry|Liquids|Electricity',
 'Final Ene

# Regional consistency checks

In [15]:

# Regroup all the rescaled countries within their regions

region_map = {
    'R10AFRICA':'SSA',
    'R10CHINA+':'CHA',
    'R10EUROPE':'EUR',
    'R10LATIN_AM':'LAM',
    'R10MIDDLE_EAST':'MEA',
    'R10REF_ECON':'REF',
    'R10REST_ASIA':'OAS',
    'R10NORTH_AM':'USA',
    'R10PAC_OECD':'JPN',
    'R10INDIA+':'IND',
    'R10ROWO':'ROWO',
    'World':'World'}


def create_macro_df(idf: pyam.IamDataFrame):
    """Converts data at country level to macro-region data"""

    # Dataset preparation
    df_country = idf.timeseries()
    df_country.index.names = ['model', 'scenario', 'country', 'variable', 'unit']
    
    # Label each country based on what region it's part of
    df_country = df_country.pix.semijoin(REMIND_RMAP.index, how="left")

    # Kick out countries which don't have a region mapping               
    df_country=df_country.loc[~isin(region=np.nan)]
    
    # # Groupby regions and sum (much faster than doing in pyam)
    df_country = pyam.IamDataFrame(
        df_country
        .groupby(['model','scenario','variable','unit','region'])
        .sum())
    
    df_region = pyam.IamDataFrame(df_country)
       
    # Drop 2023 data as it is currently missing quite a few countries (and so the regional totals are off)
    df_region.filter(year=2023,keep=False,inplace=True)

    # Rename regions
    df_region = (df_region
                 .rename(region=region_map)
                 .filter(region=['SSA','CHA','NEU','CAZ','EUR','IND','LAM','MEA','USA','JPN','REF','OAS'])
                )

    # Add world data
    for var in df_region.variable:
        df_region.aggregate_region(
            var,
            'World',
            append=True)

    return df_region

In [ ]:
raw_remind = pyam.IamDataFrame(DSCALE_PATH / "data/REMIND_data/REMIND_harm_correctionBio.csv")

FileNotFoundError: No such file: '/Users/marie-charlottegeffray/Library/CloudStorage/Box-Box/Climate Policy Team/02 - Projects/IKEA NDC 1.5° Pathways 23-25 - phase II/2 - Work Packages/WP2 - National 1.5°C Pathways/Downscaling/DSCALE/data/REMIND_data/REMIND_harm_correctionBio.csv'

In [ ]:
# Recompute macro-region from DSCALE results
macro_dscale = create_macro_df(dscale_results)

In [ ]:
# Compare with original IAM data
reg_check = subtract(
    raw_remind.timeseries(), 
    macro_dscale.timeseries()
    ).dropna(how="all")

In [ ]:
# Remove insignificant variables and data points
reg_check=reg_check.loc[~isin(region = ["JPN", "USA", "IND", "World"])].loc[~ismatch(variable = ["Price**", "Capaci**", "Trade**", "Revenu**", "Carbon**"])]
reg_check=reg_check.droplevel("unnamed: 0")
reg_check=reg_check.drop([2005,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019], axis=1)
reg_check=reg_check.dropna(axis=1)

In [ ]:
# First mask: Look at results where DSCALE results > IAMs regions (significantly)
mask = (reg_check < -1 ).sum(axis=1)
mask=mask.loc[mask>0]
len(mask)

45

In [326]:
reg_check.loc[mask.index].index.unique("variable")

Index(['Emissions|CO2', 'Primary Energy|Fossil|w/o CCS',
       'Final Energy|Transportation|Liquids|Oil', 'Primary Energy',
       'Primary Energy|Coal', 'Primary Energy|Coal|w/o CCS',
       'Primary Energy|Fossil', 'Primary Energy|Gas',
       'Primary Energy|Gas|w/o CCS', 'Primary Energy|Geothermal',
       'Primary Energy|Oil', 'Primary Energy|Oil|w/o CCS',
       'Primary Energy|Solar', 'Primary Energy|Wind',
       'Secondary Energy|Electricity', 'Secondary Energy|Electricity|Coal',
       'Secondary Energy|Electricity|Solar',
       'Secondary Energy|Electricity|Wind', 'Emissions|CO2|Energy',
       'Emissions|CO2|Industrial Processes',
       'Final Energy|Transportation|Liquids|Coal'],
      dtype='object', name='variable')

In [ ]:
# Plot to evaluate the inconsistency
ts_macro = macro_dscale.timeseries()
ts_raw = raw_remind.timeseries().droplevel("unnamed: 0")

for i in mask.index:
    fig, ax = plt.subplots()

    ts_macro.loc[[i]].T.plot(ax=ax, label = "DSCALE reaggregated")
    ts_raw.loc[[i]].T.plot(ax=ax, label = "Raw REMIND")
    ax.legend(["DSCALE reaggregated", "Raw REMIND"])


    fname = f"{i[2]}_{i[3].replace('/', '_')}.png"
    plt.savefig(f"data_checks/data_checks_neg/{fname}")
    plt.close(fig)


In [ ]:
# Then look the other way around, where IAMs regions > DSCALE results
mask = (reg_check > 1 ).sum(axis=1)
mask=mask.loc[mask>0]
len(mask)

60

In [329]:
reg_check.loc[mask.index].index.unique("variable")

Index(['Emissions|CO2', 'Emissions|CO2|Energy',
       'Emissions|CO2|Industrial Processes', 'Primary Energy|Fossil|w/o CCS',
       'Secondary Energy|Electricity', 'Secondary Energy|Hydrogen',
       'Primary Energy|Gas', 'Primary Energy|Gas|w/o CCS',
       'Primary Energy|Nuclear', 'Primary Energy', 'Primary Energy|Solar',
       'Secondary Energy|Electricity|Solar', 'Primary Energy|Fossil',
       'Final Energy|Heat', 'Secondary Energy|Heat'],
      dtype='object', name='variable')

In [317]:
ts_macro = macro_dscale.timeseries()
ts_raw = raw_remind.timeseries().droplevel("unnamed: 0")

for i in mask.index:
    fig, ax = plt.subplots()

    ts_macro.loc[[i]].T.plot(ax=ax, label = "DSCALE reaggregated")
    ts_raw.loc[[i]].T.plot(ax=ax, label = "Raw REMIND")
    ax.legend(["DSCALE reaggregated", "Raw REMIND"])


    fname = f"{i[2]}_{i[3].replace('/', '_')}.png"
    plt.savefig(f"data_checks/data_checks_pos/{fname}")
    plt.close(fig)


# Harmonisation checks - need to have downscaled more than one scenario
* Final Energy variables are not harmonised. 
* We are interested by Primary Energy, Secondary Energy and Emissions variables

In [39]:
len_scen = len(dscale_results.scenario)

In [46]:
unnecessary_variable = ["Capacity*", "Final*", "Stati*", "Price*"]

In [ ]:
base_year = 2020
harm_dict_not_equal = dict()
harm_dict_almost_equal = dict()
for variable in dscale_results.filter(variable = unnecessary_variable, keep=False).variable:
    agg = (dscale_results.filter(variable = variable, year = base_year).timeseries().groupby(["model", "region", "variable", "unit"]).sum()/ len_scen).round(2)
    core = dscale_results.filter(variable = variable, scenario = "NPE-core", year = base_year).timeseries().droplevel(["scenario"]).round(2)
    try:
        mask_equal = (agg != core)
        if len(mask_equal.loc[mask_equal[base_year]==True].index.unique("region")) > 0: 
            harm_dict_not_equal[variable] = mask_equal.loc[mask_equal[base_year]==True].index.unique("region")

        mask_almost = abs(agg - core) > 1
        if len(mask_almost.loc[mask_almost[base_year]==True].index.unique("region")) > 0:
            harm_dict_almost_equal[variable] = mask_almost.loc[mask_almost[base_year]==True].index.unique("region") 
    except: 
        print(variable)

In [45]:
list(harm_dict_not_equal.keys())

[]

In [47]:
list(harm_dict_almost_equal.keys())

[]